In [ ]:
"""
Prepare cell-cycle dataset + fixed FlowMap embedding.
Saves:
    - X_cc.npy
    - V_cc.npy
    - color_cell_cycle_relativePos.npy
    - genes_cc.txt
    - flowmap_embedding.npy
into: ./data/real_data_benchmark/cell_cycle/
"""

import os
import numpy as np
import anndata as ad

# --------------------------
# 1. Output directory
# --------------------------
outdir = "./data/real_data_benchmark/cell_cycle"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Load data
# --------------------------
adata = ad.read_h5ad("./data/fucci/rpe1_kinetics_processed.h5ad")
print("Loaded AnnData:", adata)

# --------------------------
# 3. Cell-cycle gene lists
# --------------------------
s_genes = [
    "MCM5","PCNA","TYMS","FEN1","MCM7","MCM4","RRM1","UNG","GINS2","MCM6",
    "CDCA7","DTL","PRIM1","UHRF1","CENPU","HELLS","RFC2","POLR1B","NASP",
    "RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2",
    "RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2",
    "USP1","CLSPN","POLA1","CHAF1B","MRPL36","E2F8"
]

g2m_genes = [
    "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2",
    "CKS1B","MKI67","TMPO","CENPF","TACC3","PIMREG","SMC4","CCNB2","CKAP2L",
    "CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP",
    "CDCA3","JPT1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5",
    "CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5",
    "CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"
]

cell_cycle_list = s_genes + g2m_genes

# --------------------------
# 4. Extract total expression & velocity
# --------------------------
X_all = adata.layers["X_total"].toarray()
V_all = adata.layers["velocity_T"].toarray()

# --------------------------
# 5. Gene-level filtering
# --------------------------
avg_expr = X_all.mean(axis=0)
expr_threshold = np.quantile(avg_expr, 0.2)
expr_mask = avg_expr > expr_threshold

nonzero_velocity_mask = (V_all != 0).any(axis=0)

global_mask = expr_mask & nonzero_velocity_mask

X_filtered = X_all[:, global_mask]
V_filtered = V_all[:, global_mask]
genes_filtered = np.array(adata.var_names[global_mask])

print(f"Filtered genes: {len(genes_filtered)}")

# --------------------------
# 6. Transform X and V
# --------------------------
X_log = np.log1p(X_filtered)
X_log = X_log - X_log.mean(axis=0, keepdims=True)
V_std = V_filtered / (V_filtered.std(axis=0, ddof=0) + 1e-8)

# --------------------------
# 7. Subset to cell-cycle genes
# --------------------------
genes_present = [g for g in cell_cycle_list if g in genes_filtered]
gene_indices = [np.where(genes_filtered == g)[0][0] for g in genes_present]

X_cc = X_log[:, gene_indices]
V_cc = V_std[:, gene_indices]
genes_cc = np.array(genes_present)

print("Cell-cycle matrix shapes:", X_cc.shape, V_cc.shape)

# --------------------------
# 8. Extract coloring variable
# --------------------------
color = np.array(adata.obs["Cell_cycle_relativePos"]).astype(float)

# --------------------------
# 9. Save X, V, color, genes
# --------------------------
np.save(f"{outdir}/X_cc.npy", X_cc)
np.save(f"{outdir}/V_cc.npy", V_cc)
np.save(f"{outdir}/color_cell_cycle_relativePos.npy", color)

with open(f"{outdir}/genes_cc.txt", "w") as f:
    for g in genes_cc:
        f.write(g + "\n")

print("Saved raw matrices.")

# --------------------------
# 10. Load precomputed FlowMap embedding
# --------------------------
import pickle

emb_path = "./figures/cell_cycle/flowmap_embedding.pkl"

with open(emb_path, "rb") as f:
    emb = pickle.load(f)

print("Loaded FlowMap embedding object from:", emb_path)

# --------------------------
# 11. Save the FlowMap embedding
# --------------------------
np.save(f"{outdir}/flowmap_embedding.npy", emb.X_emb)
print("Saved FlowMap embedding to flowmap_embedding.npy")

print("\nFinished preparing + embedding cell-cycle dataset.")

In [ ]:
"""
Correct pancreas preprocessing (ROBUST VERSION):
Save X_pca, V_pca (stochastic + dynamical), UMAP, and pseudotime
WITHOUT recomputing UMAP.

This version NEVER crashes on pseudotime.
If velocity pseudotime is unavailable, it falls back to diffusion pseudotime.

Outputs:
    X_pca.npy
    V_pca_stochastic.npy
    V_pca_dynamical.npy
    scvelo_umap_embedding.npy
    pseudotime.npy
"""

import scanpy as sc
import scvelo as scv

# --------------------------
# 1. Output directory
# --------------------------
outdir = "./data/real_data_benchmark/pancreas"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Load dataset
# --------------------------
adata = sc.read_h5ad(
    "./data/pancreas/pancreas_inferred_velocity.h5ad"
)
print("Loaded AnnData:", adata)

# --------------------------
# 3. Standard scVelo preprocessing
# --------------------------
scv.pp.filter_and_normalize(adata)
scv.pp.moments(adata, n_pcs=50, n_neighbors=30)

# --------------------------
# 4. Compute velocities
# --------------------------
scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity")

scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity")

# ----------------------------------------------------
# 5. Compute pseudotime (ROBUST)
# ----------------------------------------------------
pseudotime = None

# --- Try velocity pseudotime first (best case)
try:
    scv.tl.velocity_pseudotime(adata)
    if "velocity_pseudotime" in adata.obs:
        pseudotime = adata.obs["velocity_pseudotime"].to_numpy()
        print("Using velocity_pseudotime.")
except Exception as e:
    print("velocity_pseudotime failed:", str(e))

# --- Fallback: diffusion pseudotime (always works)
if pseudotime is None:
    print("Falling back to diffusion pseudotime.")
    sc.tl.diffmap(adata)
    sc.tl.dpt(adata)
    pseudotime = adata.obs["dpt_pseudotime"].to_numpy()

# Normalize to [0, 1]
pseudotime = (pseudotime - pseudotime.min()) / (
    pseudotime.max() - pseudotime.min() + 1e-8
)

np.save(f"{outdir}/pseudotime.npy", pseudotime)
print("Saved pseudotime:", pseudotime.shape)

# ----------------------------------------------------
# 6. Save ORIGINAL UMAP
# ----------------------------------------------------
X_umap = adata.obsm["X_umap"]
np.save(f"{outdir}/scvelo_umap_embedding.npy", X_umap)
print("Saved ORIGINAL UMAP:", X_umap.shape)

# ----------------------------------------------------
# 7. PCA coordinates
# ----------------------------------------------------
X_pca = adata.obsm["X_pca"]
np.save(f"{outdir}/X_pca.npy", X_pca)
print("Saved X_pca:", X_pca.shape)

# --------------------------
# 8. Stochastic velocity in PCA space
# --------------------------
if "stochastic_velocity_pca" not in adata.obsm:
    scv.tl.velocity_embedding(
        adata, basis="pca", vkey="stochastic_velocity"
    )

V_pca_stoch = adata.obsm["stochastic_velocity_pca"]
np.save(f"{outdir}/V_pca_stochastic.npy", V_pca_stoch)
print("Saved V_pca_stochastic:", V_pca_stoch.shape)

# ----------------------------------------------------
# 9. Dynamical velocity in PCA space
# ----------------------------------------------------
if "dynamical_velocity_pca" not in adata.obsm:
    scv.tl.velocity_embedding(
        adata, basis="pca", vkey="dynamical_velocity"
    )

V_pca_dyn = adata.obsm["dynamical_velocity_pca"]
np.save(f"{outdir}/V_pca_dynamical.npy", V_pca_dyn)
print("Saved V_pca_dynamical:", V_pca_dyn.shape)

print("\n✅ Finished pancreas preprocessing (UMAP untouched, pseudotime guaranteed).")

In [ ]:
adata

In [ ]:
bdata = sc.read_h5ad(
    "./data/GSE132188_adata.h5ad.h5"
)

bdata

In [ ]:
print(bdata.X[:10,:10])

In [ ]:
# =====================================================
# SAFE R EXPORT + SAVE FULL ADATA
# =====================================================

import os
import numpy as np
from scipy import sparse
from scipy.io import mmwrite

r_outdir = "./data/pancreas_R"
os.makedirs(r_outdir, exist_ok=True)

# --------------------------
# 1. Expression
# --------------------------
X = adata.X
if not sparse.issparse(X):
    X = sparse.csr_matrix(X)

mmwrite(f"{r_outdir}/expression.mtx", X)

# --------------------------
# 2. Velocity (auto detect)
# --------------------------
velocity_layer = None

for key in ["velocity", "dynamical_velocity", "stochastic_velocity"]:
    if key in adata.layers:
        velocity_layer = key
        break

if velocity_layer is None:
    raise ValueError("No velocity layer found in adata.layers")

V = adata.layers[velocity_layer]

if not sparse.issparse(V):
    V = sparse.csr_matrix(V)

mmwrite(f"{r_outdir}/velocity.mtx", V)

print(f"Using velocity layer: {velocity_layer}")

# --------------------------
# 3. Pseudotime (auto detect)
# --------------------------
pseudotime_key = None

for key in ["robust_pseudotime",
            "velocity_pseudotime",
            "dpt_pseudotime"]:
    if key in adata.obs:
        pseudotime_key = key
        break

if pseudotime_key is None:
    print("⚠ No pseudotime found. Skipping pseudotime export.")
else:
    np.savetxt(
        f"{r_outdir}/pseudotime.csv",
        adata.obs[pseudotime_key].to_numpy(),
        delimiter=","
    )
    print(f"Using pseudotime: {pseudotime_key}")

# --------------------------
# 4. Gene & Cell names
# --------------------------
np.savetxt(
    f"{r_outdir}/genes.csv",
    adata.var_names.to_numpy(),
    fmt="%s"
)

np.savetxt(
    f"{r_outdir}/cells.csv",
    adata.obs_names.to_numpy(),
    fmt="%s"
)

# --------------------------
# 5. Save FULL AnnData
# --------------------------
adata.write("./data/pancreas/pancreas_full_processed.h5ad")

print("\n✅ Export complete.")

In [ ]:
"""
Correct dentate gyrus preprocessing (CLEAN + MASKED PSEUDOTIME):

- Keeps ALL cells (no filtering)
- Preserves ORIGINAL scVelo UMAP (no recomputation)
- Computes velocities on full dataset
- Diffusion pseudotime is COMPUTED globally but
  MASKED (NaN) for unwanted cell types + UMAP outliers

Outputs:
    X_pca.npy
    V_pca_stochastic.npy
    V_pca_dynamical.npy
    scvelo_umap_embedding.npy
    pseudotime.npy
"""

import os
import numpy as np
import scvelo as scv
import scanpy as sc

# --------------------------
# 1. Output directory
# --------------------------
outdir = "./data/real_data_benchmark/dentate_gyrus"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Load dataset (WITH existing UMAP)
# --------------------------
adata = scv.datasets.dentategyrus()
print("Loaded dentate gyrus:", adata)

# --------------------------
# 3. Save ORIGINAL scVelo UMAP
# --------------------------
X_umap = adata.obsm["X_umap"]
np.save(f"{outdir}/scvelo_umap_embedding.npy", X_umap)
print("Saved ORIGINAL UMAP:", X_umap.shape)

# --------------------------
# 4. Define pseudotime exclusion mask
# --------------------------
exclude_types = {
    "Cajal-Retzius",
    "GABA-Lhx6",
    "GABA-Cnr1",
    "OPC",
    "NFOL",
    "OL",
    "Microglia",
    "PVM",
    "VLMC",
    "Endothelial",
    "Pericytes",
}

# cell-type mask
valid_type_mask = ~adata.obs["clusters_enlarged"].isin(exclude_types)

# UMAP-1 outlier mask (top-3)
umap1 = X_umap[:, 0]
outlier_idx = np.argsort(umap1)[-3:]
outlier_mask = np.ones(adata.n_obs, dtype=bool)
outlier_mask[outlier_idx] = False

# final mask: valid for pseudotime
valid_pseudotime_mask = valid_type_mask & outlier_mask

print(
    f"Pseudotime-valid cells: {valid_pseudotime_mask.sum()} / {adata.n_obs}"
)

# --------------------------
# 5. Standard scVelo preprocessing
# --------------------------
scv.pp.filter_and_normalize(
    adata, min_shared_counts=30, n_top_genes=2000
)
scv.pp.moments(adata, n_pcs=50, n_neighbors=30)

# --------------------------
# 6. Velocity computation
# --------------------------
print("Computing stochastic velocity...")
scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity")

print("Computing dynamical velocity...")
scv.tl.recover_dynamics(adata)
scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity")

# --------------------------
# 7. Diffusion pseudotime (masked)
# --------------------------
adata.uns["iroot"] = 0  # deterministic

sc.tl.diffmap(adata)
sc.tl.dpt(adata)

pseudotime = adata.obs["dpt_pseudotime"].to_numpy()

# normalize globally
pseudotime = (pseudotime - pseudotime.min()) / (
    pseudotime.max() - pseudotime.min() + 1e-8
)

# mask invalid cells
pseudotime_masked = pseudotime.copy()
pseudotime_masked[~valid_pseudotime_mask] = np.nan

np.save(f"{outdir}/pseudotime.npy", pseudotime_masked)
print("Saved masked pseudotime:", pseudotime_masked.shape)

# --------------------------
# 8. PCA coordinates
# --------------------------
X_pca = adata.obsm["X_pca"]
np.save(f"{outdir}/X_pca.npy", X_pca)
print("Saved X_pca:", X_pca.shape)

# --------------------------
# 9. Stochastic velocity (PCA)
# --------------------------
if "stochastic_velocity_pca" not in adata.obsm:
    scv.tl.velocity_embedding(
        adata, basis="pca", vkey="stochastic_velocity"
    )

V_pca_stoch = adata.obsm["stochastic_velocity_pca"]
np.save(f"{outdir}/V_pca_stochastic.npy", V_pca_stoch)
print("Saved V_pca_stochastic:", V_pca_stoch.shape)

# --------------------------
# 10. Dynamical velocity (PCA)
# --------------------------
if "dynamical_velocity_pca" not in adata.obsm:
    scv.tl.velocity_embedding(
        adata, basis="pca", vkey="dynamical_velocity"
    )

V_pca_dyn = adata.obsm["dynamical_velocity_pca"]
np.save(f"{outdir}/V_pca_dynamical.npy", V_pca_dyn)
print("Saved V_pca_dynamical:", V_pca_dyn.shape)

print("\n✅ Finished dentate gyrus preprocessing (cells kept, pseudotime masked).")

In [ ]:
import scvelo as scv
import matplotlib.pyplot as plt

# Load dataset
adata = scv.datasets.dentategyrus()

# Basic sanity check
print(adata)

# Plot UMAP colored by cell type
scv.pl.umap(
    adata,
    color="clusters_enlarged",
    legend_loc="on data",
    frameon=False,
    size=30,
)

plt.show()

In [ ]:
import scvelo as scv
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
adata = scv.datasets.dentategyrus()

# Previously removed cell types
remove_types = {
    "Cajal-Retzius",
    "GABA-Lhx6",
    "GABA-Cnr1",
    "OPC",
    "NFOL",
    "OL",
    "Microglia",
    "PVM",
    "VLMC",
    "Endothelial",
    "Pericytes",
}

# Subset by cell type
mask = ~adata.obs["clusters_enlarged"].isin(remove_types)
adata_sub = adata[mask].copy()

# --------------------------------
# Remove top-3 largest UMAP-1 points
# --------------------------------
umap = adata_sub.obsm["X_umap"]
umap1 = umap[:, 0]

# indices of 3 largest UMAP-1 values
bad_idx = np.argsort(umap1)[-3:]

# keep everything else
keep_mask = np.ones(adata_sub.n_obs, dtype=bool)
keep_mask[bad_idx] = False

adata_sub = adata_sub[keep_mask].copy()

print("Removed cells (UMAP-1 extreme):", bad_idx)
print("Remaining cells:", adata_sub.n_obs)

# ----------------
# Plot UMAP
# ----------------
scv.pl.umap(
    adata_sub,
    color="clusters_enlarged",
    legend_loc="on data",
    frameon=False,
    size=35,
)

plt.show()

In [ ]:
"""
Prepare Larry hematopoiesis dataset for FlowMap benchmarking
(raw gene space + FlowMap embedding).

Outputs saved into:
    ./data/real_data_benchmark/larry/

Saves:
  - X_raw.npy
  - V_raw.npy
  - color_pseudotime.npy        # visualization-only color
  - flowmap_embedding.npy
  - scvelo_processed.h5ad       # still contains velocity_pseudotime
"""

import os
import numpy as np
import anndata as ad
import scvelo as scv
from sklearn.preprocessing import StandardScaler
import joblib

# --------------------------
# 1. Output directory
# --------------------------
outdir = "./data/real_data_benchmark/larry"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Load dataset
# --------------------------
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
print("Loaded Larry dataset:", adata)

# --------------------------
# 3. Extract raw X
# --------------------------
if "spliced" in adata.layers:
    X = adata.layers["spliced"].copy()
    print("Using adata.layers['spliced'] as raw X.")
else:
    X = adata.X.copy()
    print("Using adata.X as raw X.")

if not isinstance(X, np.ndarray):
    X = X.toarray()

# HVG restriction
hvg_mask = adata.var["highly_variable"].values
X = X[:, hvg_mask]

# Log1p + scale
X = np.log1p(X)
scaler_x = StandardScaler(with_mean=True, with_std=True)
X = scaler_x.fit_transform(X)

# --------------------------
# 4. Extract + clean raw V
# --------------------------
if "velocity" in adata.layers:
    V = adata.layers["velocity"].copy()
elif "velocity_pyro" in adata.layers:
    V = adata.layers["velocity_pyro"].copy()
else:
    raise ValueError("Missing velocity layer (expected 'velocity' or 'velocity_pyro').")

if not isinstance(V, np.ndarray):
    V = V.toarray()

V = V[:, hvg_mask]

# Fix NaNs (Larry has many)
gene_means = np.nanmean(V, axis=0)
gene_means[np.isnan(gene_means)] = 0.0
inds = np.where(np.isnan(V))
V[inds] = np.take(gene_means, inds[1])

# Variance-scale velocities
scaler_v = StandardScaler(with_mean=False, with_std=True)
V = scaler_v.fit_transform(V)

print(
    f"X shape: {X.shape}\n"
    f"V shape: {V.shape}\n"
    f"NaNs in V after cleanup: {np.isnan(V).sum()}"
)

# --------------------------
# 5. Save raw matrices
# --------------------------
np.save(f"{outdir}/X_raw.npy", X)
np.save(f"{outdir}/V_raw.npy", V)

# --------------------------
# 6. Compute velocity-based pseudotime (KEEP IT)
# --------------------------
scv.tl.velocity_pseudotime(adata)
print("Computed velocity_pseudotime (kept in AnnData).")

# --------------------------
# 7. Load FlowMap embedder (NO recomputation)
# --------------------------
emb = joblib.load("./data/larry/larry_embedder.pkl")
print("Loaded FlowMap embedder:", type(emb))

X_emb = emb.X_emb

# --------------------------
# 8. Dumb visualization pseudotime (distance from ref)
# --------------------------
# Dumb reference point (given)
ref = np.array([3.0868282, 3.5019479])

# Euclidean distance from ref
distance_larry = np.linalg.norm(X_emb - ref[None, :], axis=1)

# Normalize to [0, 1] (just for nicer colors)
distance_larry = (distance_larry - distance_larry.min()) / (
    distance_larry.max() - distance_larry.min() + 1e-8
)

np.save(f"{outdir}/distance_pseudotime.npy", distance_larry)
print("Saved visualization pseudotime color:", distance_larry.shape)

# --------------------------
# 9. Save FlowMap embedding
# --------------------------
np.save(f"{outdir}/flowmap_embedding.npy", X_emb)
print("Saved FlowMap embedding:", X_emb.shape)

# --------------------------
# 10. Save processed AnnData
# --------------------------
adata.write_h5ad(f"{outdir}/scvelo_processed.h5ad", compression="gzip")

print(f"\nFinished processing Larry dataset. Saved to: {outdir}/")